In [28]:
import pandas as pd
from rapidfuzz import fuzz

batch_cooking = pd.read_csv("batch_cooking_recipes.csv")
breakfast = pd.read_csv("breakfast_recipes.csv")
dinner = pd.read_csv("dinner_recipes.csv")
lunch = pd.read_csv("lunch_recipes.csv")
quick_and_easy = pd.read_csv("quick_and_easy_recipes.csv")

lunch.head()

,title,url,ingredient,quantity,note,kcal,fat,saturates,carbs,sugars,fibre,protein,salt
0,No title,https://www.bbcgoodfood.com/recipes/falafel-bu...,chickpeas,400g can,rinsed and drained,175,8,1,18.0,1.0,4,6,0.4
1,No title,https://www.bbcgoodfood.com/recipes/falafel-bu...,red onion,1 small,roughly chopped,175,8,1,18.0,1.0,4,6,0.4
2,No title,https://www.bbcgoodfood.com/recipes/falafel-bu...,garlic clove,1,chopped,175,8,1,18.0,1.0,4,6,0.4
3,No title,https://www.bbcgoodfood.com/recipes/falafel-bu...,flat-leaf parsley,handful of,or curly parsley,175,8,1,18.0,1.0,4,6,0.4
4,No title,https://www.bbcgoodfood.com/recipes/falafel-bu...,ground cumin,1 tsp,NaN,175,8,1,18.0,1.0,4,6,0.4


In [29]:
breakfast["meal_type"] = "breakfast"
lunch["meal_type"] = "lunch"
dinner["meal_type"] = "dinner"
batch_cooking["meal_type"] = "batch cooking"

recipes_df = pd.concat([breakfast, lunch, dinner, batch_cooking], ignore_index=True)

In [30]:
recipes = []

for url, group in recipes_df.groupby("url"):
    recipe = {}

    recipe["title"] = group["title"].iloc[0]
    recipe["url"] = url
    recipe["meal_type"] = group["meal_type"].iloc[0]

    recipe["ingredients"] = [
        {
            "ingredient": row["ingredient"],
            "quantity": row["quantity"],
            "note": row["note"]
        }
        for _, row in group.iterrows()
    ]

    nutrition_cols = ["kcal","fat","saturates","carbs","sugars","fibre","protein","salt"]
    recipe["nutrition"] = group[nutrition_cols].iloc[0].to_dict()

    recipes.append(recipe)

print("Total recipes:", len(recipes))

Total recipes: 395


In [34]:
print("\nRecipe Assistant (type 'exit' to stop)\n")

while True:
    user_input = input("What ingredients do you have? ").strip()
    
    if user_input.lower() == "exit":
        break

    user_ingredients = [i.strip().lower() for i in user_input.split(",")]

    meal_type = input("Meal type? (breakfast/lunch/dinner or enter to skip): ").strip().lower()
    if meal_type == "":
        meal_type = None

    # filter recipes
    filtered_recipes = recipes
    if meal_type:
        filtered_recipes = [r for r in recipes if r["meal_type"] == meal_type]

    scored = []

    for r in filtered_recipes:
        recipe_ings = [i["ingredient"].lower() for i in r["ingredients"]]

        matches = []

        for ui in user_ingredients:
            for ri in recipe_ings:
                if fuzz.partial_ratio(ui, ri) >= 80:
                    matches.append(ri)
                    break

        score = len(matches) / len(recipe_ings) if recipe_ings else 0

        if score > 0:
            scored.append((score, matches, r))

    if not scored:
        print("\n❌ No matching recipes found.\n")
        continue

    scored.sort(reverse=True, key=lambda x: x[0])

    best_score, matched_ings, recipe = scored[0]

    print(f"\n✅ Best match: {recipe['title']} ({recipe['meal_type']})")
    print(f"Match score: {round(best_score, 2)}")
    print(f"You have: {matched_ings}")

    # Build prompt
    recipe_ingredients = [i["ingredient"] for i in recipe["ingredients"]]

    prompt = f"""
You are a helpful cooking assistant.

User has these ingredients:
{", ".join(user_ingredients)}

Recipe: {recipe['title']} ({recipe['meal_type']})

Ingredients in recipe:
{", ".join(recipe_ingredients)}

User already has:
{", ".join(matched_ings)}

Tasks:
1. List missing ingredients
2. Suggest substitutions if possible
3. Provide simple step-by-step instructions
4. Keep it concise
"""

    use_llm = input("\nGenerate full recipe with LLM? (y/n): ").lower()

    if use_llm == "y":
        import ollama

        response = ollama.chat(
            model="llama3"
            messages=[
                {"role": "user", "content": prompt}
            ]
        )

print(response["message"]["content"])

    print("\n" + "="*50 + "\n")

SyntaxError: invalid syntax. Perhaps you forgot a comma? (1257534714.py, line 80)